# 🩺 MedGraphRAG — Interactive Experimentation Notebook

A **notebook-driven control panel** for the `clinical_graphrag_study` repository
(Hybrid Graph-RAG for Medical QA). It clones the repo, wires every pipeline stage
behind a one-line wrapper, and runs the offline-validated paths end-to-end on Kaggle.

**What this notebook gives you**

| Stage | Wrapper | Repo modules used |
|---|---|---|
| 1 · Manifest & budget | `load_manifest / budget_summary` | `manifest.manifest` |
| 2 · Data prep | `make_synth` / MIRAGE converter | `mgr.data.*` |
| 3 · Generation backend | `make_gen_client(CFG)` | `mgr.clients.*` (+ local HF) |
| 4 · Smoke → gate **H2** | `run_smoke_stage()` | `mgr.smoke`, `mgr.runner` |
| 5 · Retrieval metrics | `bm25.retrieve` + `metrics.retrieval` | `mgr.retrieval.*` |
| 6 · UMLS grounding (**F5**) | `UMLSLinker` + coverage | `mgr.graph.umls` |
| 7 · CA-RRF fusion (**C2**) | `ca_rrf(use_concept=…)` | `mgr.retrieval.ca_rrf` |
| 8 · CARe gate (**C3**, **F4**) | `CareGate.fit` + frontier | `mgr.rerank.care_gate` |
| 9 · RGD decomposition (**C1**, **F3**) | `decompose` | `mgr.stats.rgd` |
| 10 · Statistics | audit → CI → exact-p → Holm → effect size | `mgr.stats.*`, `mgr.eval.*` |

> **The three contributions** — C1 Retrieval-Generation Decomposition, C2 Concept-Aware RRF,
> C3 Cost-Aware adaptive Reranking — each get a dedicated cell.

### ⚙️ Kaggle setup before you run
1. **Settings → Internet → ON** (needed to `git clone` the repo).
2. **Accelerator → GPU** is *optional* — only the `local_hf` generation backend uses it.
   The default `heuristic` backend runs fully on CPU with no model download.
3. Everything writes under `/kaggle/working/mgr_runs/` so artifacts persist as notebook output.

> ☁️ **Full-scale note:** the real 244-run sweep needs a RunPod A100 serving a 70B model + a
> NIM key (see the repo's `docs/RUNBOOK.md`). Kaggle can't host that, so this notebook exercises
> the harness, retrieval, grounding, fusion, gate, decomposition, stats, and figures on a small
> in-notebook benchmark — the same code paths, miniature scale. Point `GEN_BACKEND="vllm"` at a
> served endpoint to scale up unchanged.

## 1 · Configuration

Everything you'd want to change lives here. Edit, then **Run All**. Re-running individual
cells is safe — each stage caches its artifacts and skips recomputation unless `CFG.FORCE=True`.

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class Config:
    # ---- repository ----
    REPO_URL: str    = "https://github.com/SLIMIHamda/clinical_graphrag_study.git"
    REPO_BRANCH: str = "main"
    REPO_DIR: str    = ""   # resolved in the environment cell
    RUN_DIR: str     = ""   # resolved in the environment cell

    # ---- experiment selection (must exist in the workbook) ----
    BENCHMARK: str = "MMLU-Med"     # MMLU-Med | MedQA-US | MedMCQA | PubMedQA | BioASQ-YN | ...
    BACKBONE: str  = "Llama-70B"    # Llama-70B | Qwen-14B | Mistral-24B | BioMed-8B
    SMOKE_SEED: int = 42            # 42 | 123 | 7
    N_ITEMS: int    = 24            # items in the in-notebook smoke benchmark

    # ---- generation backend ----
    # "heuristic": offline deterministic stand-in (no GPU/net) — default, always works
    # "local_hf" : small HuggingFace instruct model on the Kaggle GPU
    # "vllm"     : an OpenAI-compatible vLLM server (the real 70B path)
    # "openai"   : any OpenAI-compatible endpoint
    GEN_BACKEND: str  = "heuristic"
    HF_MODEL_ID: str  = "Qwen/Qwen2.5-0.5B-Instruct"
    MAX_NEW_TOKENS: int = 8
    TEMPERATURE: float  = 0.0
    VLLM_BASE_URL: str  = "http://localhost:8000"
    API_KEY: str        = ""

    # ---- statistics (modest for notebook speed; raise for real reporting) ----
    N_BOOT: int = 10_000
    N_PERM: int = 100_000

    # ---- reproducibility & control ----
    SEED: int = 0
    RUN_TESTS: bool = False        # run the repo's 139 pytest cases
    INSTALL_EDITABLE: bool = False # pip install -e . (slower; not required)
    FORCE: bool = False            # recompute every stage, ignoring caches

CFG = Config()
print("Configuration:")
for k, v in asdict(CFG).items():
    print(f"  {k:16s} = {v!r}")

## 2 · Environment, paths, seeds & logging

Detects Kaggle vs local, builds the organized output tree, pins seeds, and sets up logging
to both stdout and a file.

In [ ]:
import os, sys, json, logging, random
from pathlib import Path
import numpy as np

KAGGLE = Path("/kaggle/working").is_dir()
BASE = Path("/kaggle/working") if KAGGLE else Path("./mgr_kaggle").resolve()
CFG.REPO_DIR = str(BASE / "clinical_graphrag_study")
CFG.RUN_DIR  = str(BASE / "mgr_runs")
RUN = Path(CFG.RUN_DIR)

# organized artifact tree (Kaggle best practice: everything under working dir)
DIRS = {k: RUN / k for k in ["data", "results", "figures", "artifacts", "logs", "checkpoints"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# reproducibility
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
try:
    import torch
    torch.manual_seed(CFG.SEED)
    HAS_GPU = torch.cuda.is_available()
    GPU = torch.cuda.get_device_name(0) if HAS_GPU else "none"
except Exception:
    HAS_GPU, GPU = False, "torch-not-installed"

# logging (force=True so re-running this cell re-initialises cleanly)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(DIRS["logs"] / "run.log")],
    force=True,
)
log = logging.getLogger("mgr")
log.info(f"Kaggle={KAGGLE}  GPU={GPU}  BASE={BASE}")

print(f"Kaggle env : {KAGGLE}")
print(f"GPU        : {GPU}")
print("Output tree:")
for k, v in DIRS.items():
    print(f"  {k:12s} -> {v}")

## 3 · Clone the repository & install dependencies

Idempotent: re-running won't re-clone. If Internet is off but you attached the repo as a Kaggle
**Dataset**, it is detected under `/kaggle/input`. The repo's HTTP clients are pure-stdlib, so the
dependency footprint is light.

In [ ]:
import subprocess, shutil, sys
from pathlib import Path

def sh(cmd, **kw):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, text=True, **kw)

repo = Path(CFG.REPO_DIR)
already = (repo / "pyproject.toml").exists()
if not already:
    r = sh(f"git clone --depth 1 -b {CFG.REPO_BRANCH} {CFG.REPO_URL} '{repo}'")
    if r.returncode != 0:
        cand = list(Path("/kaggle/input").glob("**/pyproject.toml")) if Path("/kaggle/input").exists() else []
        if cand:
            src = cand[0].parent
            print("Clone failed; using attached dataset copy at", src)
            shutil.copytree(src, repo, dirs_exist_ok=True)
        else:
            raise RuntimeError(
                "Clone failed and no /kaggle/input copy found. "
                "Enable Internet (Settings → Internet → ON) or attach the repo as a Dataset."
            )
else:
    print("Repository already present:", repo)

# light runtime deps (most are preinstalled on Kaggle)
sh(f"{sys.executable} -m pip install -q openpyxl pyyaml numpy duckdb pyarrow matplotlib tqdm pandas")
if CFG.INSTALL_EDITABLE:
    sh(f"{sys.executable} -m pip install -q -e '{repo}'")

# make the repo importable without an editable install
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import mgr, manifest  # noqa: F401
print("Imported mgr + manifest from", repo)

In [ ]:
# Optional: run the repository's own test suite (139 cases) as a sanity check.
if CFG.RUN_TESTS:
    import subprocess, sys
    subprocess.run(f"{sys.executable} -m pytest -q", shell=True, cwd=CFG.REPO_DIR)
else:
    print("Skipping repo test suite. Set CFG.RUN_TESTS=True to run all 139 tests.")

## 4 · Shared helpers

Small utilities used by every stage: JSON I/O, a cache check, and rich display shortcuts.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Markdown, Image

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))
    return path

def load_json(path):
    return json.loads(Path(path).read_text())

def cached(path, force=None):
    """True if the artifact exists and we are not forcing recomputation."""
    force = CFG.FORCE if force is None else force
    return (not force) and Path(path).exists()

def show_df(df, n=10):
    display(df.head(n))
    return df

def show_img(path):
    display(Image(str(path)))

print("Helpers ready: save_json, load_json, cached, show_df, show_img, tqdm")

## 5 · Stage 1 — Manifest & budget reconciliation

The workbook (`Experiment-Matrix.xlsx`) is the source of truth: 244 runs = condition × benchmark ×
backbone × seed. This validates the design invariants and reproduces the **$591.49 / $887.23**
budget envelope exactly from the cost model.

In [ ]:
from manifest.manifest import load_manifest, validate, budget_summary
import pandas as pd, matplotlib.pyplot as plt

M = load_manifest()
problems = validate(M)
bs = budget_summary(M)

print(f"Manifest      : {len(M)} runs")
print(f"Validation    : {len(problems)} problems {problems[:3]}")
print(f"Budget (base) : ${bs['est_cost_usd']:.2f}")
print(f"Budget (1.5x) : ${bs['est_cost_usd_1.5x']:.2f}")
print(f"Est. tokens   : {bs['est_tokens']/1e6:.0f}M")

rows = [dict(run_id=r.run_id, phase=r.phase, condition=r.condition, benchmark=r.benchmark,
             backbone=r.backbone, seed=r.seed, n_items=r.n_items,
             est_cost=round(r.computed_est_cost(), 2), gate=r.gate, status=r.status)
        for r in M.runs]
df_manifest = pd.DataFrame(rows)
df_manifest.to_csv(DIRS["artifacts"] / "manifest_runs.csv", index=False)

by_cond = df_manifest.groupby("condition")["est_cost"].sum().sort_values(ascending=False)
display(by_cond.to_frame("est_cost_usd"))
ax = by_cond.plot.barh(figsize=(7, 4), title="Estimated API cost by condition (USD)")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

show_df(df_manifest)

## 6 · Stage 2 — Data preparation

Builds a small, self-contained medical-MCQ benchmark **plus a matching passage corpus**, written
in the repo's normalized JSONL schema (`{qid, question, options, answer}` and `{id, text}`). Each
question's correct fact appears in exactly one corpus passage, so BM25 retrieval is *useful* — the
retrieval-augmented arm should beat the closed-book arm, giving an interpretable smoke result.

> If you attach the real **MIRAGE `benchmark.json`** as a Kaggle dataset, it is detected and
> converted via `mgr.data.convert_mirage` instead.

In [ ]:
import random, json
from pathlib import Path

data_root   = DIRS["data"]
bench_path  = data_root / f"{CFG.BENCHMARK}.jsonl"
corpus_path = data_root / "corpus.jsonl"
rel_path    = DIRS["artifacts"] / "relevance.json"

def make_synth(n, seed):
    # Each question carries a unique key token (qNNNN). Its corpus passage shares ONLY that
    # key plus the correct option's unique tokens — and no generic English words the question
    # uses — so BM25 retrieves exactly the right passage and the RAG arm can answer from context
    # while the closed-book arm cannot. This makes the smoke result interpretable by construction.
    rng = random.Random(seed)
    TOPICS = ["hypertension", "sepsis", "myocardial infarction", "asthma", "diabetes mellitus",
              "pneumonia", "stroke", "anemia", "hypothyroidism", "pulmonary embolism",
              "atrial fibrillation", "cirrhosis", "meningitis", "glaucoma", "osteoporosis"]
    letters = "ABCD"
    items, corpus, rel = [], [], {}
    for i in range(n):
        key, topic = f"q{i:04d}", rng.choice(TOPICS)
        # option text = key + a per-option unique nonce token (no shared English words)
        opts = {letters[j]: f"{key} choice{j} {key}c{j}n{rng.randrange(1000, 9999)}"
                for j in range(4)}
        ci = rng.randrange(4)
        q = (f"For case {key} regarding {topic}, which option is supported by the record?")
        items.append({"qid": key, "question": q, "options": opts, "answer": letters[ci]})
        pid = f"p{i:04d}"
        # passage = key + the correct option verbatim, nothing else (no generic tokens)
        corpus.append({"id": pid, "text": f"{key} {opts[letters[ci]]}"})
        rel[key] = [pid]
    return items, corpus, rel

if cached(bench_path) and cached(corpus_path) and cached(rel_path):
    print("Reusing cached dataset.")
else:
    mirage = list(Path("/kaggle/input").glob("**/benchmark.json")) if Path("/kaggle/input").exists() else []
    used_mirage = False
    if mirage:
        try:
            from mgr.data.convert_mirage import convert_file
            written = convert_file(mirage[0], data_root)
            print("Converted MIRAGE:", written)
            if bench_path.exists():
                its = [json.loads(l) for l in bench_path.read_text().splitlines() if l][:CFG.N_ITEMS]
                corpus = [{"id": f"p{i:04d}", "text": it.get("question", "")} for i, it in enumerate(its)]
                rel = {it["qid"]: [f"p{i:04d}"] for i, it in enumerate(its)}
                with open(corpus_path, "w") as fh:
                    for c in corpus: fh.write(json.dumps(c) + "\n")
                save_json(rel, rel_path)
                used_mirage = True
        except Exception as e:
            print("MIRAGE conversion failed, using synthetic:", e)
    if not used_mirage:
        items, corpus, rel = make_synth(CFG.N_ITEMS, CFG.SEED)
        with open(bench_path, "w") as fh:
            for it in items: fh.write(json.dumps(it) + "\n")
        with open(corpus_path, "w") as fh:
            for c in corpus: fh.write(json.dumps(c) + "\n")
        save_json(rel, rel_path)
    print("Wrote", bench_path.name, "and", corpus_path.name)

items  = [json.loads(l) for l in bench_path.read_text().splitlines() if l]
corpus = [json.loads(l) for l in corpus_path.read_text().splitlines() if l]
RELEVANCE = load_json(rel_path)
print(f"Benchmark={CFG.BENCHMARK}  items={len(items)}  corpus_docs={len(corpus)}")
show_df(pd.DataFrame(items)[["qid", "question", "answer"]], 5)

## 7 · Stage 3 — Generation backend (pluggable)

`make_gen_client(CFG)` returns an object implementing the repo's
`complete_text(model, messages, **params) -> (text, token_usage)` protocol. Swap backends with one
config line — the rest of the pipeline never changes (prompt parity is preserved).

- **heuristic** — deterministic, offline; for MCQ it lexically matches the options against the
  retrieved context (so the BM25 arm meaningfully outperforms closed-book). Zero downloads.
- **local_hf** — a small instruct model on the Kaggle GPU.
- **vllm / openai** — wraps the repo's real clients for a served endpoint.

In [ ]:
import re, hashlib

class HeuristicClient:
    """Offline deterministic LLM stand-in. Implements complete_text."""
    def _hash_pick(self, text, choices):
        h = int(hashlib.sha256(text.encode()).hexdigest(), 16)
        return choices[h % len(choices)]

    def complete_text(self, model, messages, **params):
        user = messages[-1]["content"]
        toks_in = max(1, len(user) // 4)
        ctx = ""
        if "Context:" in user:
            ctx = user.split("Context:", 1)[1].split("Question:", 1)[0]
        opts = dict(re.findall(r"^([A-D])\.\s*(.+)$", user, flags=re.M))
        if opts:
            letters = list(opts.keys())
            if ctx.strip():
                cset = set(re.findall(r"[a-z0-9]+", ctx.lower()))
                def overlap(t):
                    return len(set(re.findall(r"[a-z0-9]+", t.lower())) & cset)
                best = max(letters, key=lambda L: (overlap(opts[L]), L))
                if overlap(opts[best]) > 0:
                    return best, {"in": toks_in, "out": 1}
            return self._hash_pick(user, letters), {"in": toks_in, "out": 1}
        if "yes, no, or maybe" in user:
            return self._hash_pick(user, ["yes", "no", "maybe"]), {"in": toks_in, "out": 1}
        if "yes or no" in user:
            return self._hash_pick(user, ["yes", "no"]), {"in": toks_in, "out": 1}
        return "unknown", {"in": toks_in, "out": 1}


class LocalHFClient:
    """Wraps a small HuggingFace instruct model (Kaggle GPU). Implements complete_text."""
    def __init__(self, model_id, max_new_tokens=8, temperature=0.0):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype="auto",
            device_map="auto" if torch.cuda.is_available() else None,
        )
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def complete_text(self, model, messages, **params):
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = self.tok(prompt, return_tensors="pt").to(self.model.device)
        out = self.model.generate(
            **enc, max_new_tokens=self.max_new_tokens,
            do_sample=self.temperature > 0, temperature=max(self.temperature, 1e-5),
            pad_token_id=self.tok.eos_token_id,
        )
        gen = out[0][enc["input_ids"].shape[1]:]
        text = self.tok.decode(gen, skip_special_tokens=True)
        return text, {"in": int(enc["input_ids"].shape[1]), "out": int(gen.shape[0])}


def make_gen_client(cfg):
    b = cfg.GEN_BACKEND
    if b == "heuristic":
        log.info("Gen backend: heuristic (offline)")
        return HeuristicClient()
    if b == "local_hf":
        log.info(f"Gen backend: local_hf {cfg.HF_MODEL_ID}")
        return LocalHFClient(cfg.HF_MODEL_ID, cfg.MAX_NEW_TOKENS, cfg.TEMPERATURE)
    if b == "vllm":
        from mgr.clients.vllm import VLLMClient
        return VLLMClient(base_url=cfg.VLLM_BASE_URL, api_key=cfg.API_KEY or None)
    if b == "openai":
        from mgr.clients.openai_compat import OpenAICompatClient
        class _C(OpenAICompatClient):
            def complete_text(self, model, messages, **p):
                r = self.chat(model, messages, **p)
                u = r.get("usage", {})
                return (r["choices"][0]["message"]["content"],
                        {"in": int(u.get("prompt_tokens", 0)), "out": int(u.get("completion_tokens", 0))})
        return _C(base_url=cfg.VLLM_BASE_URL, api_key=cfg.API_KEY or None)
    raise ValueError(f"unknown GEN_BACKEND: {b}")

GEN = make_gen_client(CFG)
demo_msgs = [
    {"role": "system", "content": "test"},
    {"role": "user", "content": "Question:\npick one\n\nOptions:\nA. foo\nB. bar\nC. baz\nD. qux\n"
                                 "Respond with a single capital letter (A, B, C, or D) and nothing else."},
]
print("Backend:", CFG.GEN_BACKEND, "| sample completion ->", GEN.complete_text("demo", demo_msgs))

## 8 · Stage 4 — Smoke run (No-RAG + BM25) → gate **H2**

The Step-1 go/no-go. Runs both baseline arms through the **real** `runner → RAGExecutor →
generate → extract → score` loop for `(BENCHMARK, SMOKE_SEED, BACKBONE)`, writes qid-keyed per-item
records, and — on success — flips gate **H2**, which is what unblocks the baseline rows in the
full program.

In [ ]:
import shutil, json
from manifest.manifest import load_manifest
from mgr.smoke import run_smoke, report
from mgr.tracking import record as rec

M = load_manifest()
corpus_records = [json.loads(l) for l in (DIRS["data"] / "corpus.jsonl").read_text().splitlines() if l]
smoke_root  = DIRS["results"] / "smoke"
done_marker = DIRS["checkpoints"] / "smoke.done"

def run_smoke_stage(force=False):
    if cached(done_marker, force):
        print("Smoke already done; reusing records.")
    else:
        if smoke_root.exists():
            shutil.rmtree(smoke_root)   # so re-runs don't trip the 'already Done' guard
        records = run_smoke(
            M, GEN, data_root=str(DIRS["data"]), corpus_records=corpus_records,
            benchmark=CFG.BENCHMARK, seed=CFG.SMOKE_SEED, backbone=CFG.BACKBONE,
            n_items=CFG.N_ITEMS, results_root=str(smoke_root),
        )
        report(records)
        done_marker.write_text("ok")
    out = []
    for cond in ("No-RAG", "BM25"):
        row = next(r for r in M.runs if r.condition == cond and r.benchmark == CFG.BENCHMARK
                   and r.seed == CFG.SMOKE_SEED and r.backbone == CFG.BACKBONE)
        out.append(rec.read_record(row.run_id, str(smoke_root)))
    return out

SMOKE = run_smoke_stage(force=CFG.FORCE)

df_smoke = pd.DataFrame([{
    "run_id": r.run_id, "condition": r.condition, "n_items": r.n_items,
    "accuracy": r.metrics.get("generation", {}).get("accuracy"),
    "coverage": r.metrics.get("generation", {}).get("coverage"),
    "tokens": r.tokens.get("total"), "status": r.status,
} for r in SMOKE])
display(df_smoke)

H2_OK = all(r.status == "Done" for r in SMOKE)
GATE_LEDGER = {"H2": H2_OK, "G3": False, "P3": False}
save_json(GATE_LEDGER, DIRS["artifacts"] / "gate_ledger.json")
print("✅ Gate H2 satisfied:", H2_OK, "→ baseline rows would unblock." if H2_OK else "→ investigate failures.")

## 9 · Stage 5 — Retrieval metrics (Recall@k · Precision@k · MRR · nDCG)

Scores BM25 retrieval against the gold passage per question using the repo's metric library.
Recall@3 is the workbook's primary retrieval metric and feeds the RGD decomposition in Stage 9.

In [ ]:
from mgr.retrieval.factory import build_retriever
from mgr.metrics import retrieval as rmetrics

bm25 = build_retriever("BM25", corpus_records)
RELEVANCE = load_json(DIRS["artifacts"] / "relevance.json")
items = [json.loads(l) for l in (DIRS["data"] / f"{CFG.BENCHMARK}.jsonl").read_text().splitlines() if l]

pairs = []
for it in tqdm(items, desc="BM25 retrieve"):
    res = bm25.retrieve(it["question"], depth_k=10)
    pairs.append((res.retrieved_ids, RELEVANCE.get(it["qid"], [])))

scores = rmetrics.score(pairs)
df_ret = pd.DataFrame({"Recall@k": scores.recall, "Precision@k": scores.precision})
display(df_ret)
print(f"MRR = {scores.mrr:.3f}   nDCG@10 = {scores.ndcg.get(10):.3f}")
save_json({"recall": scores.recall, "precision": scores.precision,
           "mrr": scores.mrr, "ndcg": scores.ndcg}, DIRS["artifacts"] / "retrieval_scores.json")

## 10 · Stage 6 — UMLS grounding & coverage curve (Figure **F5**)

Links entity mentions to CUIs by **exact → abbreviation → fuzzy** matching and emits the coverage
curve — the control for the thesis "~47% ungrounded" defect. Each tier recovers more mentions.

In [ ]:
from mgr.graph.umls import UMLSLinker, coverage
from mgr.figures import plot_coverage_curve

exact_map = {"myocardial infarction": "C0027051", "hypertension": "C0020538", "sepsis": "C0243026",
             "asthma": "C0004096", "stroke": "C0038454", "pneumonia": "C0032285", "anemia": "C0002871"}
abbrev_map = {"mi": "myocardial infarction", "htn": "hypertension", "cva": "stroke", "cap": "pneumonia"}
linker = UMLSLinker(exact_map=exact_map, abbrev_map=abbrev_map, fuzzy_threshold=0.85)

mentions = ["myocardial infarction", "MI", "HTN", "hypertention", "sepsis",
            "CVA", "pnumonia", "unknownterm", "asthma", "anemia"]
links = linker.link_mentions(mentions)
display(pd.DataFrame([{"term": l.term, "cui": l.cui, "method": l.method} for l in links]))

cov = coverage(mentions, linker)
print(f"Coverage: exact={cov.exact} abbrev={cov.abbrev} fuzzy={cov.fuzzy} "
      f"unlinked={cov.unlinked} / total={cov.total}")
fig_path = plot_coverage_curve(cov.curve, DIRS["figures"] / "F5_coverage.png")
show_img(fig_path)

## 11 · Stage 7 — CA-RRF fusion · contribution **C2**

Concept-Aware Reciprocal Rank Fusion adds a UMLS-concept-overlap list as a first-class ranking
signal. The key property is **isolability**: `use_concept=False` is *exactly* plain RRF over the
same components — so the ablation measures only the concept list's marginal value. Watch a
concept-heavy doc that's lexically buried get rescued.

In [ ]:
from mgr.retrieval.ca_rrf import ca_rrf, concept_ranked_list
from mgr.retrieval.rrf import reciprocal_rank_fusion

component_lists = {"lexical": ["d1", "d2", "d3", "d4", "d5"],
                   "dense":   ["d2", "d1", "d6", "d3", "d5"]}
query_concepts = {"C0027051", "C0020538"}
candidate_concepts = {"d5": {"C0027051", "C0020538"}, "d1": {"C0004096"}, "d2": set(),
                      "d3": {"C0027051"}, "d4": set(), "d6": set()}

def order(fused):
    return [d for d, _ in fused]

plain  = reciprocal_rank_fusion(list(component_lists.values()), k=60)
abl    = ca_rrf(component_lists, query_concepts, candidate_concepts, k=60, use_concept=False)
carrf  = ca_rrf(component_lists, query_concepts, candidate_concepts, k=60, use_concept=True)

print("plain RRF      :", order(plain))
print("CA-RRF (off)   :", order(abl), " == plain RRF  ← the isolable ablation")
print("CA-RRF (on)    :", order(carrf))
print("concept list   :", concept_ranked_list(query_concepts, candidate_concepts))
print(f"\nd5: RRF rank #{order(plain).index('d5')+1}  →  CA-RRF rank #{order(carrf).index('d5')+1}"
      "  (concept overlap promoted it)")

## 12 · Stage 8 — CARe gate & cost–quality frontier · contribution **C3** (Figure **F4**)

A per-query gate predicts whether cross-encoder reranking will help, so compute is spent only on
ambiguous queries. We simulate per-query fusion-score profiles + with/without-rerank outcomes,
derive **oracle** labels, fit the logistic gate, and plot **CARe vs always vs never** on the
cost–quality frontier.

In [ ]:
import numpy as np
from mgr.rerank.care_gate import extract_features, oracle_benefit, CareGate, cost_quality_frontier
from mgr.figures import plot_pareto

rng = np.random.default_rng(CFG.SEED)
feats, labels, q_with, q_wo = [], [], [], []
for _ in range(300):
    ambiguous = rng.random() < 0.5
    if ambiguous:                                   # near-tie → reranking tends to help
        scores = sorted(rng.uniform(0.4, 0.6, size=6), reverse=True)
        wo = rng.binomial(1, 0.45); wr = min(1, wo + rng.binomial(1, 0.5))
    else:                                           # clear winner → reranking rarely helps
        scores = sorted(np.concatenate([[0.95], rng.uniform(0.1, 0.4, size=5)]), reverse=True)
        wo = rng.binomial(1, 0.8); wr = wo
    feats.append(extract_features(list(scores), query_type=float(ambiguous)))
    q_wo.append(float(wo)); q_with.append(float(wr)); labels.append(oracle_benefit(wr, wo))

gate = CareGate.fit(feats, labels)
decisions = [gate.decide(f, value=1.0, cost=0.15) for f in feats]
frontier = cost_quality_frontier(decisions, q_with, q_wo, rerank_cost=1.0)

display(pd.DataFrame([{
    "policy": k, "rerank_rate": round(v.rerank_rate, 3),
    "mean_quality": round(v.mean_quality, 3), "total_cost": v.total_cost,
} for k, v in frontier.items()]))
show_img(plot_pareto(frontier, DIRS["figures"] / "F4_pareto.png"))

## 13 · Stage 9 — Retrieval–Generation Decomposition · contribution **C1** (Figure **F3**)

The headline object: per system, *retrieval gain* vs *generation gain* against a baseline, on the
same items. A system whose retrieval improves but whose answers don't (`diverges=True`) is the
exact phenomenon C1 was built to expose — an illustrative `Graph-only` point is included to show it.

In [ ]:
from mgr.stats.rgd import decompose, divergent_systems
from mgr.figures import plot_rgd

ret = load_json(DIRS["artifacts"] / "retrieval_scores.json")
recall3 = ret["recall"].get("3") or list(ret["recall"].values())[1]
acc = {r.condition: r.metrics.get("generation", {}).get("accuracy", 0.0) for r in SMOKE}

systems = {
    "No-RAG":     (0.0,     acc.get("No-RAG", 0.0)),
    "BM25":       (recall3, acc.get("BM25", 0.0)),
    "Graph-only": (0.92,    acc.get("No-RAG", 0.0) - 0.02),   # illustrative divergent point
}
pts = decompose(systems, baseline="No-RAG")
display(pd.DataFrame([{
    "system": p.system, "retrieval": round(p.retrieval, 3), "generation": round(p.generation, 3),
    "ret_gain": round(p.retrieval_gain, 3), "gen_gain": round(p.generation_gain, 3),
    "diverges": p.diverges,
} for p in pts]))
print("Divergent (retrieval ↑, generation flat/↓):", divergent_systems(pts))
show_img(plot_rgd(pts, DIRS["figures"] / "F3_rgd.png", baseline="No-RAG"))

## 14 · Stage 10 — Statistics pipeline

The full inferential chain on the smoke arms, joined by `qid`:
**answer-format audit** (gates EM/F1) → **bootstrap CI** → **paired exact-p permutation** (refuses
the resolution floor) → **Holm–Bonferroni** → **effect sizes** (Cohen's d, Cliff's δ). The audit
runs *first* and withholds EM/F1 if the arms aren't comparable — the control for the
"EM 0→0.76" artifact.

In [ ]:
import json, numpy as np
from mgr.eval.answer_format_audit import audit
from mgr.stats.bootstrap import bootstrap_diff_ci
from mgr.stats.permutation import paired_permutation_test
from mgr.stats.holm import holm_bonferroni
from mgr.stats.effect_size import cohens_d_paired, cliffs_delta, interpret_cliffs

def load_items_jsonl(record):
    return [json.loads(l) for l in open(record.items_path) if l.strip()]

by = {r.condition: {it["qid"]: it for it in load_items_jsonl(r)} for r in SMOKE}
qids = sorted(set(by["No-RAG"]) & set(by["BM25"]))
a = np.array([by["BM25"][q]["correct"] for q in qids], dtype=float)
b = np.array([by["No-RAG"][q]["correct"] for q in qids], dtype=float)

# 1) answer-format audit gates EM/F1
arm_labels = {c: [by[c][q]["answer_norm"] for q in qids] for c in ("No-RAG", "BM25")}
rep = audit(arm_labels, answer_type="mcq")
print("Answer-format audit passed:", rep.gate_emit_em_f1(), "| reasons:", rep.reasons)

if rep.gate_emit_em_f1():
    ci = bootstrap_diff_ci(a, b, n_boot=CFG.N_BOOT, seed=CFG.SEED)
    pr = paired_permutation_test(a, b, n_perm=CFG.N_PERM, seed=CFG.SEED)
    holm = holm_bonferroni({"BM25_vs_NoRAG": pr.p_value})
    d, delta = cohens_d_paired(a, b), cliffs_delta(a, b)
    print(f"accuracy   BM25={a.mean():.3f}  No-RAG={b.mean():.3f}  Δ={a.mean()-b.mean():+.3f}")
    print(f"95% CI(Δ)  [{ci.lo:+.3f}, {ci.hi:+.3f}]  (n_boot={ci.n_boot})")
    print(f"perm test  p={pr.p_value:.2e}  exact={pr.exact}  at_floor={pr.at_floor}  (n_perm={pr.n_perm})")
    for h in holm:
        print(f"Holm       {h.label}: p_adj={h.p_adj:.2e}  reject={h.reject}")
    print(f"effect     Cohen's d={d:.3f}   Cliff's δ={delta:.3f} ({interpret_cliffs(delta)})")
    save_json({"delta": float(a.mean()-b.mean()), "ci": [ci.lo, ci.hi],
               "p": pr.p_value, "at_floor": pr.at_floor, "cohens_d": d, "cliffs_delta": delta},
              DIRS["artifacts"] / "stats.json")
else:
    print("EM/F1 withheld — audit failed (this is the intended guard, not an error).")

## 15 · Figures gallery

All figures emitted by the stages above, collected in one place.

In [ ]:
from pathlib import Path
figs = sorted(Path(DIRS["figures"]).glob("*.png"))
print("Figures:", [f.name for f in figs])
for f in figs:
    show_img(f)

## 16 · Run summary & artifact manifest

Everything produced this session, organized under `/kaggle/working/mgr_runs/`.

In [ ]:
from pathlib import Path
rows = []
for d in DIRS.values():
    for f in Path(d).rglob("*"):
        if f.is_file():
            rows.append({"artifact": str(f.relative_to(RUN)), "bytes": f.stat().st_size})
df_art = pd.DataFrame(rows).sort_values("artifact").reset_index(drop=True)
display(df_art)

summary = {
    "benchmark": CFG.BENCHMARK, "backbone": CFG.BACKBONE, "n_items": CFG.N_ITEMS,
    "gen_backend": CFG.GEN_BACKEND, "H2_satisfied": bool(GATE_LEDGER["H2"]),
    "n_artifacts": len(df_art), "run_dir": str(RUN),
}
save_json(summary, DIRS["artifacts"] / "run_summary.json")
print("\nStages completed: manifest · data · smoke(H2) · retrieval · grounding(F5) · "
      "CA-RRF(C2) · CARe(C3,F4) · RGD(C1,F3) · stats")
print("Summary:", summary)

## 17 · Scaling to the full program (cloud)

This notebook runs the harness at miniature scale. The real 244-run sweep is a cloud job — the code
is identical, only the resources change:

1. **Generation** → set `CFG.GEN_BACKEND="vllm"` and `CFG.VLLM_BASE_URL` to a served
   `meta/llama-3.3-70b-instruct` (AWQ-int4 on one A100, per the repo's Decision D3).
2. **Embeddings / reranker / RAGAS judge** → a NIM free-tier key (`NIM_API_KEY`); never used for
   generation.
3. **Graph build → gate G3**, then the **sweep**, **oracle labels → gate P3**, and final stats —
   follow `docs/RUNBOOK.md` in the cloned repo. `infra/runpod/with_pod.sh` guarantees pod teardown.

The gate ledger this notebook wrote (`artifacts/gate_ledger.json`) mirrors what the real pipeline
flips as each go/no-go criterion is met.